In [1]:
from google.colab import drive 
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/MorDIFF

/content/drive/MyDrive/MorDIFF


In [3]:
# Define datasets we are going to be working with, for readablity
DATASET_PATH = "/content/drive/MyDrive/Datasets/"
ALIGNED_PATH = "/content/drive/MyDrive/Datasets_aligned"

In [ ]:
%%bash
# Install pre-compiled stuff for dlib
apt-get update -qq
apt-get install -y cmake build-essential

pip install dlib-bin
# Install packages which are not usually pre-installed in Colab
pip install -q "pytorch-lightning<2.0" pytorch-fid lpips lmdb facenet-pytorch

Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 238 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 59.3 MB/s eta 0:00:00


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
# align dataset images while preserving the original structure
import os
import glob
import bz2
import requests
import numpy as np
import scipy.ndimage
import PIL.Image
import dlib
from tqdm import tqdm

# --- 1. Automated Model Weight Downloader ---
def get_shape_predictor():
    url = 'http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2'
    bz2_path = 'temp/shape_predictor_68_face_landmarks.dat.bz2'
    dat_path = 'temp/shape_predictor_68_face_landmarks.dat'
    
    if os.path.exists(dat_path):
        return dat_path
        
    os.makedirs('temp', exist_ok=True)
    if not os.path.exists(bz2_path):
        print("Downloading dlib 68-face landmarks model...")
        response = requests.get(url, stream=True)
        with open(bz2_path, 'wb') as f:
            f.write(response.content)
            
    print("Decompressing landmarks model...")
    with open(dat_path, 'wb') as f_out, bz2.BZ2File(bz2_path, 'rb') as f_in:
        f_out.write(f_in.read())
        
    return dat_path

# --- 2. FFHQ Standard Image Alignment Transform ---
def image_align(src_file, dst_file, face_landmarks, output_size=256, transform_size=4096, enable_padding=True):
    lm = np.array(face_landmarks)
    lm_eye_left = lm[36:42]
    lm_eye_right = lm[42:48]
    lm_mouth_outer = lm[48:60]

    eye_left = np.mean(lm_eye_left, axis=0)
    eye_right = np.mean(lm_eye_right, axis=0)
    eye_avg = (eye_left + eye_right) * 0.5
    eye_to_eye = eye_right - eye_left
    mouth_left = lm_mouth_outer[0]
    mouth_right = lm_mouth_outer[6]
    mouth_avg = (mouth_left + mouth_right) * 0.5
    eye_to_mouth = mouth_avg - eye_avg

    x = eye_to_eye - np.flipud(eye_to_mouth) * [-1, 1]
    x /= np.hypot(*x)
    x *= max(np.hypot(*eye_to_eye) * 2.0, np.hypot(*eye_to_mouth) * 1.8)
    y = np.flipud(x) * [-1, 1]
    c = eye_avg + eye_to_mouth * 0.1
    quad = np.stack([c - x - y, c - x + y, c + x + y, c + x - y])
    qsize = np.hypot(*x) * 2

    img = PIL.Image.open(src_file).convert('RGB')

    shrink = int(np.floor(qsize / output_size * 0.5))
    if shrink > 1:
        rsize = (int(np.rint(float(img.size[0]) / shrink)), int(np.rint(float(img.size[1]) / shrink)))
        img = img.resize(rsize, PIL.Image.LANCZOS)
        quad /= shrink
        qsize /= shrink

    border = max(int(np.rint(qsize * 0.1)), 3)
    crop = (int(np.floor(min(quad[:, 0]))), int(np.floor(min(quad[:, 1]))),
            int(np.ceil(max(quad[:, 0]))), int(np.ceil(max(quad[:, 1]))))
    crop = (max(crop[0] - border, 0), max(crop[1] - border, 0),
            min(crop[2] + border, img.size[0]), min(crop[3] + border, img.size[1]))
    if crop[2] - crop[0] < img.size[0] or crop[3] - crop[1] < img.size[1]:
        img = img.crop(crop)
        quad -= crop[0:2]

    pad = (int(np.floor(min(quad[:, 0]))), int(np.floor(min(quad[:, 1]))),
           int(np.ceil(max(quad[:, 0]))), int(np.ceil(max(quad[:, 1]))))
    pad = (max(-pad[0] + border, 0), max(-pad[1] + border, 0),
           max(pad[2] - img.size[0] + border, 0), max(pad[3] - img.size[1] + border, 0))
    if enable_padding and max(pad) > border - 4:
        pad = np.maximum(pad, int(np.rint(qsize * 0.3)))
        img = np.pad(np.float32(img), ((pad[1], pad[3]), (pad[0], pad[2]), (0, 0)), 'reflect')
        h, w, _ = img.shape
        y_grid, x_grid, _ = np.ogrid[:h, :w, :1]
        mask = np.maximum(
            1.0 - np.minimum(np.float32(x_grid) / pad[0], np.float32(w - 1 - x_grid) / pad[2]),
            1.0 - np.minimum(np.float32(y_grid) / pad[1], np.float32(h - 1 - y_grid) / pad[3])
        )
        blur = qsize * 0.02
        img += (scipy.ndimage.gaussian_filter(img, [blur, blur, 0]) - img) * np.clip(mask * 3.0 + 1.0, 0.0, 1.0)
        img += (np.median(img, axis=(0, 1)) - img) * np.clip(mask, 0.0, 1.0)
        img = PIL.Image.fromarray(np.uint8(np.clip(np.rint(img), 0, 255)), 'RGB')
        quad += pad[:2]

    img = img.transform((transform_size, transform_size), PIL.Image.QUAD, (quad + 0.5).flatten(), PIL.Image.BILINEAR)
    if output_size < transform_size:
        img = img.resize((output_size, output_size), PIL.Image.LANCZOS)

    img.save(dst_file, 'PNG')

# --- 3. Recursive Dataset Pipeline ---
def align_dataset(input_root, output_root):
    model_path = get_shape_predictor()
    detector = dlib.get_frontal_face_detector()
    predictor = dlib.shape_predictor(model_path)

    # Recursively gather images across all subfolders
    extensions = ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG')
    image_paths = []
    for ext in extensions:
        image_paths.extend(glob.glob(os.path.join(input_root, '**', ext), recursive=True))

    print(f"Discovered {len(image_paths)} images across dataset tree.")

    for img_path in tqdm(image_paths, desc="Aligning Faces"):
        rel_path = os.path.relpath(img_path, input_root)
        name_without_ext = os.path.splitext(rel_path)[0]
        out_path = os.path.join(output_root, f"{name_without_ext}.png")

        if os.path.exists(out_path):
            continue

        os.makedirs(os.path.dirname(out_path), exist_ok=True)

        try:
            img = dlib.load_rgb_image(img_path)
            dets = detector(img, 1)
            if len(dets) == 0:
                continue

            # Align using primary detected face
            landmarks = [(p.x, p.y) for p in predictor(img, dets[0]).parts()]
            image_align(img_path, out_path, landmarks, output_size=256)
        except Exception as e:
            print(f"Skipping {img_path}: {e}")

    print(f"Alignment completed. Files saved to: {output_root}")

# --- Execution ---
align_dataset(
    input_root=DATASET_PATH, 
    output_root=f"{ALIGNED_PATH}/{DATASET_PATH.split('/')[-1]}"
)

Discovered 321 images across dataset tree.


Aligning Faces:   0%|          | 0/321 [00:00<?, ?it/s]/tmp/ipykernel_359/1246277963.py:92: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PIL.Image.fromarray(np.uint8(np.clip(np.rint(img), 0, 255)), 'RGB')
Aligning Faces: 100%|██████████| 321/321 [09:30<00:00,  1.78s/it]

Alignment completed. Files saved to: /content/drive/MyDrive/Datasets_aligned/


In [11]:
# Get cosine similarity between all images and determine which once are most similar and will produce a good enough morph. 
import os
# we are going to have this block executed only if there are not morph guidelines already generated for this dataset
# because it costs a lot of time to compute
if not os.path.exists(f"/content/drive/MyDrive/Morph_guidelines/{DATASET_PATH.split('/')[-1]}/morph_guideines.csv"):
    import torch
    import numpy as np
    import pandas as pd
    from facenet_pytorch.models.inception_resnet_v1 import InceptionResnetV1
    from PIL import Image
    from torchvision import transforms
    from tqdm import tqdm

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

    transform = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])

    # 1. Compute embeddings for all aligned images
    aligned_dir = f"/content/drive/MyDrive/Datasets_aligned/{DATASET_PATH.split('/')[-1]}"
    image_files = [os.path.join(root, f) 
                for root, _, files in os.walk(aligned_dir) 
                for f in files if f.lower().endswith(('.png', '.jpg'))]

    embeddings = []
    valid_files = []

    with torch.no_grad():
        for f in tqdm(image_files, desc="Extracting embeddings"):
            img = Image.open(f).convert('RGB')
            tensor = transform(img).unsqueeze(0).to(device)
            emb = resnet(tensor).cpu().numpy().flatten()
            emb = emb / np.linalg.norm(emb)  # L2 Normalize
            embeddings.append(emb)
            valid_files.append(f)

    embeddings = np.array(embeddings)

    # 2. Compute Cosine Similarity Matrix
    similarity_matrix = np.dot(embeddings, embeddings.T)
    np.fill_diagonal(similarity_matrix, -1) # Exclude self-matching

    # 3. Select top matches (Similarity threshold e.g. > 0.5 or nearest neighbor)
    morph_pairs = []
    used_indices = set()

    for i in range(len(valid_files)):
        if i in used_indices:
            continue
        best_match_idx = np.argmax(similarity_matrix[i])
        sim_score = similarity_matrix[i][best_match_idx]
        
        # Require decent baseline similarity for a viable morph
        if sim_score >= 0.40 and best_match_idx not in used_indices:
            morph_pairs.append({
                "img1": valid_files[i],
                "img2": valid_files[best_match_idx],
                "similarity": float(sim_score)
            })
            used_indices.add(i)
            used_indices.add(best_match_idx)

    # Save guidelines to CSV
    df_pairs = pd.DataFrame(morph_pairs)
    df_pairs.to_csv(f"/content/drive/MyDrive/Morph_guidelines/{DATASET_PATH.split('/')[-1]}/morph_guidelines.csv", index=False)
    print(f"Generated {len(df_pairs)} optimal morphing pairs -> morph_guidelines.csv")

else: 
    print(f"Morph guidelines already exist for {DATASET_PATH.split('/')[-1]}. Skipping generation.")

AttributeError: partially initialized module 'torchvision' has no attribute 'extension' (most likely due to a circular import)

In [ ]:
# define dataset morphing function using DiffAE model
import os
import sys
import torch
from PIL import Image
from torchvision import transforms
from torch.functional import F
import matplotlib.pyplot as plt
from tqdm import tqdm

path_to_diff_model = "./diffae-master"
sys.path.append(path_to_diff_model)
from templates import *

def cos_sim(a, b):
    a = F.normalize(a.view(-1), dim=0)
    b = F.normalize(b.view(-1), dim=0)
    return (a * b).sum()

def morph_dataset(pairs_list, output_dir, device='cuda:0'):
    os.makedirs(output_dir, exist_ok=True)

    # 1. Load Model ONCE into VRAM
    print("Loading DiffAE Model...")
    conf = ffhq256_autoenc()
    model = LitModel(conf)
    state = torch.load(
        f'{path_to_diff_model}/checkpoints/{conf.name}/last.ckpt',
        map_location='cpu',
        weights_only=False
    )
    model.load_state_dict(state['state_dict'], strict=False)
    model.ema_model.eval()
    model.ema_model.to(device)

    image_size = conf.img_size
    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    to_pil = transforms.ToPILImage()

    def load_aligned(path):
        img = Image.open(path).convert('RGB')
        if img.size[0] != image_size or img.size[1] != image_size:
            img = img.resize((image_size, image_size), Image.LANCZOS)
        return transform(img)

    alpha = torch.tensor([0.0, 0.5, 1.0]).to(device)

    # 2. Iterate over all image pairs rapidly
    print(f"Starting batch morphing for {len(pairs_list)} pairs...")
    with torch.no_grad():
        for img1_path, img2_path in tqdm(pairs_list):
            name1 = os.path.splitext(os.path.basename(img1_path))[0]
            name2 = os.path.splitext(os.path.basename(img2_path))[0]
            out_file = os.path.join(output_dir, f"morphed_{name1}_and_{name2}.png")

            if os.path.exists(out_file):
                continue

            img1 = load_aligned(img1_path)
            img2 = load_aligned(img2_path)
            batch = torch.stack([img1, img2]).to(device)

            cond = model.encode(batch)
            xT = model.encode_stochastic(batch, cond, T=250)

            intp = cond[0][None] * (1 - alpha[:, None]) + cond[1][None] * alpha[:, None]
            theta = torch.arccos(cos_sim(xT[0], xT[1]))
            x_shape = xT[0].shape

            intp_x = (
                torch.sin((1 - alpha[:, None]) * theta) * xT[0].flatten(0, 2)[None] +
                torch.sin(alpha[:, None] * theta) * xT[1].flatten(0, 2)[None]
            ) / torch.sin(theta)
            intp_x = intp_x.view(-1, *x_shape)

            pred = model.render(intp_x, intp, T=20)
            morph_img = to_pil(pred[1].cpu())
            morph_img.save(out_file)

cached
cached
total img files 2
output aligned images at: diff/aligned/
Model params: 160.69 M


100%|██████████| 2/2 [00:02<00:00,  1.00s/it]
Global seed set to 0
/content/drive/MyDrive/MorDIFF/diffae-master/diffusion/base.py:306: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(self.conf.fp16):
